In [ ]:
import torch
import numpy as np
import torch.nn.functional as F_conv
import matplotlib.pyplot as plt
import torchvision.transforms.functional as F

import pinn_starlight_core.data.PhotoLoader as Loader

# ============ 1. 读取图片，转灰度 ============
loader = Loader.RAWLoader()
loader.load(r"../../data/real_raw/origin/03.jpg")
gray_img = np.mean(loader.rgb_data, axis=2).astype(np.float32)   # (H, W) 灰度 numpy
H, W = gray_img.shape
# 转成 (1, 1, H, W) 给 PyTorch 卷积用: (batch, channel, height, width)
img_tensor = torch.from_numpy(gray_img)[None, None]

# ============ 2. 高斯模糊 → 抹掉星点，留光污染 ============
kernel_size = 71
sigma = kernel_size / 3.0   # 经验法则: sigma ≈ kernel/3
blurred = F.gaussian_blur(img_tensor, [kernel_size, kernel_size],
                          sigma=[sigma, sigma]).squeeze()   # → (H, W)

# ============ 3. 拉普拉斯 ∇² → 找亮度变化最剧烈的地方 ============
# 拉普拉斯核: 中心 -4, 上下左右各 +1, 加起来=0
# 在均匀区域响应=0, 在凸起(光源中心)响应为负, 在凹陷(暗斑中心)响应为正
laplacian_kernel = torch.tensor([[0., 1., 0.],
                                  [1., -4., 1.],
                                  [0., 1., 0.]])
laplacian = F_conv.conv2d(blurred[None, None],
                          laplacian_kernel.view(1, 1, 3, 3),
                          padding=1).squeeze()   # padding=1 保持尺寸不变

# ============ 4. 三重门限: 只保留地平线附近的"真光源"信号 ============

# (a) 亮度门限: 只留最亮的 30% 区域 (地平线/城市灯光)
bright_mask = (blurred > blurred.quantile(0.70)).float()

# (b) 垂直衰减: 天顶权重=0, 地平线权重=1
# 光污染随高度指数衰减, 星云不受这规律约束 → 用这个压星云
y_axis = torch.linspace(0, 1, H)              # 0=图顶(天顶), 1=图底(地平线)
vertical_decay = (1 - torch.exp(-y_axis * 6))[:, None]   # (H, 1) → 自动广播到 (H, W)

# (c) 边缘置零: 砍掉梯度幅度 top 5% 的像素 (建筑剪影边缘/星点核心)
# 这些地方拉普拉斯会爆, 但不是光污染源
gy, gx = torch.gradient(blurred)
gradient_magnitude = torch.sqrt(gx**2 + gy**2)
edge_mask = (gradient_magnitude < gradient_magnitude.quantile(0.95)).float()

# 三重 mask 相乘
icity = laplacian * bright_mask * vertical_decay * edge_mask

# ============ 5. 显示 ============
# 用 98% 分位裁剪极端值, 否则图被极值拉成一片白
display_range = np.percentile(icity.abs().numpy(), 98)

fig, ax = plt.subplots(1, 4, figsize=(18, 4))
ax[0].imshow(gray_img, cmap='gray');       ax[0].set_title('原图')
ax[1].imshow(blurred, cmap='gray');        ax[1].set_title(f'高斯模糊 k={kernel_size}')
ax[2].imshow(laplacian, cmap='RdBu',
             vmin=-display_range, vmax=display_range);  ax[2].set_title('拉普拉斯 ∇²')
ax[3].imshow(icity, cmap='RdBu',
             vmin=-display_range, vmax=display_range);  ax[3].set_title('I_city (三重 mask 后)')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()